# Gemma 4 31B-it on Colab L4 (24GB) — 4-bit NF4
Runtime > Change runtime type > **L4 GPU**. bf16 needs ~62GB, so we quantize to 4-bit (~18GB).
Accept the license on the model page and add `HF_TOKEN` to Colab Secrets if the repo is gated.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes torchvision

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
try: login(userdata.get('HF_TOKEN'))
except Exception as e: print('No HF_TOKEN secret, continuing anonymously:', e)

from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
MODEL_ID = 'google/gemma-4-31B-it'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')
print(f'GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
def chat(prompt, thinking=False, max_new_tokens=512):
    messages = [{'role':'system','content':'You are a helpful assistant.'},
                {'role':'user','content':prompt}]
    inputs = processor.apply_chat_template(messages, tokenize=True, return_dict=True,
        return_tensors='pt', add_generation_prompt=True, enable_thinking=thinking).to(model.device)
    n = inputs['input_ids'].shape[-1]
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=1.0, top_p=0.95, top_k=64)
    return processor.decode(out[0][n:], skip_special_tokens=True)

print(chat('سلام! یک جوک کوتاه بگو.'))